# 26. 경제 시계열 예측용 데이터셋 만들기

이번 장에서는 25장에서 저장한 FRED 경제 지표 CSV를 예측용 데이터셋으로 바꿉니다.

목표:

```text
현재/과거 경제 지표 -> 다음 달 CPI 예측
```

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

## 2. 저장된 FRED CSV 읽기

25장 노트북을 실행하면 아래 경로에 CSV가 저장됩니다.

```text
C:\work\deepLearning\deepLearning_textbook\data_outputs\fred_economic_indicators.csv
```

In [ ]:
DATA_PATH = Path(r"C:\work\deepLearning\deepLearning_textbook\data_outputs\fred_economic_indicators.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "FRED 경제 지표 CSV가 없습니다. 먼저 25장 노트북을 실행해 CSV를 저장하세요."
    )

# parse_dates는 date 컬럼을 읽을 때 바로 datetime으로 변환합니다.
df = pd.read_csv(DATA_PATH, parse_dates=["date"])

df = df.sort_values("date")

print("데이터 shape:", df.shape)
df.head()

## 3. 데이터 기본 확인

In [ ]:
print("컬럼 목록:", list(df.columns))
print("\n데이터 타입:")
print(df.dtypes)
print("\n결측치 개수:")
print(df.isna().sum())
print("\n날짜 범위:", df["date"].min(), "~", df["date"].max())

## 4. 예측 target 만들기

이번 장의 정답은 다음 달 CPI입니다.

`shift(-1)`을 사용하면 다음 행의 CPI를 현재 행에 붙일 수 있습니다.

In [ ]:
# target_next_cpi는 다음 달 CPI입니다.
# 현재 행의 입력으로 다음 달 CPI를 맞히는 문제가 됩니다.
df["target_next_cpi"] = df["cpi"].shift(-1)

df[["date", "cpi", "target_next_cpi"]].head()

## 5. lag feature 만들기

과거 값을 입력 특성으로 추가합니다.

예를 들어 `cpi_lag_1`은 1개월 전 CPI입니다.

In [ ]:
base_features = ["cpi", "unemployment_rate", "fed_funds_rate"]
lag_months = [1, 2, 3]

for feature in base_features:
    for lag in lag_months:
        # shift(lag)는 lag개월 전 값을 현재 행에 붙입니다.
        df[f"{feature}_lag_{lag}"] = df[feature].shift(lag)

df.head()

## 6. 결측치 제거

shift를 사용하면 맨 앞과 맨 뒤에 결측치가 생깁니다.

예를 들어 1월에는 1개월 전 값이 없고, 마지막 달에는 다음 달 CPI가 없습니다.

In [ ]:
print("제거 전 shape:", df.shape)
print("결측치 개수:")
print(df.isna().sum())

model_df = df.dropna().copy()

print("\n제거 후 shape:", model_df.shape)
model_df.head()

## 7. 입력 X와 정답 y 나누기

In [ ]:
target_column = "target_next_cpi"

# date와 target은 입력 특성에서 제외합니다.
feature_columns = [col for col in model_df.columns if col not in ["date", target_column]]

X = model_df[feature_columns]
y = model_df[target_column]

print("feature_columns:")
print(feature_columns)
print("\nX shape:", X.shape)
print("y shape:", y.shape)

## 8. 시간 순서 train/validation split

In [ ]:
split_index = int(len(model_df) * 0.8)

X_train = X.iloc[:split_index]
X_val = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_val = y.iloc[split_index:]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("train 날짜:", model_df["date"].iloc[0], "~", model_df["date"].iloc[split_index - 1])
print("validation 날짜:", model_df["date"].iloc[split_index], "~", model_df["date"].iloc[-1])

## 9. 스케일링

In [ ]:
scaler = StandardScaler()

# scaler는 학습 데이터에만 fit합니다.
X_train_scaled = scaler.fit_transform(X_train)

# 검증 데이터에는 transform만 적용합니다.
X_val_scaled = scaler.transform(X_val)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:", X_val_scaled.shape)

## 10. 최종 데이터셋 확인

In [ ]:
print("입력 특성 개수:", X_train_scaled.shape[1])
print("학습 샘플 수:", X_train_scaled.shape[0])
print("검증 샘플 수:", X_val_scaled.shape[0])

print("\n첫 번째 학습 입력:")
print(X_train.iloc[0])

print("\n첫 번째 학습 정답 다음 달 CPI:")
print(y_train.iloc[0])

## 11. 예측용 데이터 저장

In [ ]:
OUTPUT_DIR = Path(r"C:\work\deepLearning\deepLearning_textbook\data_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "fred_cpi_forecasting_dataset.csv"

# 모델링용 데이터셋을 CSV로 저장합니다.
model_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)

## 12. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. 경제 시계열도 시간 순서를 지켜야 한다.
2. target_next_cpi는 다음 달 CPI를 현재 행의 정답으로 붙인 것이다.
3. lag feature는 과거 값을 현재 행의 입력으로 붙인 것이다.
4. shift(-1)은 미래 target을 만들 때 쓰고, shift(1)은 과거 lag를 만들 때 쓴다.
5. train/validation split과 scaler fit도 시간 순서를 지켜야 한다.
```

## 과제

1. `target_next_cpi`가 왜 마지막 행에서 결측치가 되는지 설명해보세요.
2. `cpi_lag_1`과 `cpi_lag_3`의 차이를 설명해보세요.
3. `feature_columns`에 어떤 컬럼이 들어갔는지 확인하고 의미를 적어보세요.
4. CPI 대신 실업률을 target으로 삼으면 어떤 부분을 바꿔야 할지 적어보세요.